In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from mealpy.swarm_based import PSO
from mealpy.utils.problem import FloatVar
import warnings
warnings.filterwarnings('ignore')

# ----------------------------
# Load your Sentiment dataset
# ----------------------------
def load_data():
    amazon = pd.read_csv("amazon_cells_labelled.txt", sep="\t", header=None, names=["text", "label"])
    imdb   = pd.read_csv("imdb_labelled.txt", sep="\t", header=None, names=["text", "label"])
    yelp   = pd.read_csv("yelp_labelled.txt", sep="\t", header=None, names=["text", "label"])

    df = pd.concat([amazon, imdb, yelp], axis=0)

    vectorizer = TfidfVectorizer(stop_words="english", max_features=1000)
    X = vectorizer.fit_transform(df["text"]).toarray()
    y = df["label"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    return X_train, X_test, y_train, y_test

# ----------------------------
# Objective function for PSO
# ----------------------------
def objective_function(solution):
    global X_train, y_train

    n_estimators = int(solution[0])
    max_depth = int(solution[1]) if solution[1] > 0 else None
    min_samples_split = int(solution[2])
    min_samples_leaf = int(solution[3])
    max_features = solution[4]

    try:
        rf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=42,
            n_jobs=-1
        )
        scores = cross_val_score(rf, X_train, y_train, cv=3, scoring='accuracy')
        fitness = -np.mean(scores)  # Negative because optimizer minimizes
    except Exception:
        fitness = 1.0  # Penalty if invalid

    return fitness

# ----------------------------
# Run PSO optimization
# ----------------------------
def optimize_random_forest():
    global X_train, X_test, y_train, y_test
    X_train, X_test, y_train, y_test = load_data()

    print("Starting Random Forest optimization with Mealpy PSO...")
    print(f"Training set size: {X_train.shape}")
    print(f"Test set size: {X_test.shape}")
    print("-" * 50)

    # Problem bounds for hyperparameters
    problem = {
        "bounds": [
            FloatVar(lb=10, ub=200, name="n_estimators"),
            FloatVar(lb=1, ub=20, name="max_depth"),
            FloatVar(lb=2, ub=20, name="min_samples_split"),
            FloatVar(lb=1, ub=10, name="min_samples_leaf"),
            FloatVar(lb=0.1, ub=1.0, name="max_features")
        ],
        "minmax": "min",
        "obj_func": objective_function
    }

    optimizer = PSO.OriginalPSO(epoch=5, pop_size=10)
    best_agent = optimizer.solve(problem)   # returns dict, not Agent

    # ✅ Correct way to extract results
    best_position = best_agent["solution"]
    best_fitness = best_agent["target.fitness"]

    print("\nOptimization Results:")
    print("-" * 50)
    print(f"Best fitness (negative accuracy): {best_fitness:.6f}")
    print(f"Best accuracy: {-best_fitness:.6f}")

    best_params = {
        'n_estimators': int(best_position[0]),
        'max_depth': int(best_position[1]) if best_position[1] > 0 else None,
        'min_samples_split': int(best_position[2]),
        'min_samples_leaf': int(best_position[3]),
        'max_features': best_position[4],
        'random_state': 42
    }

    print("\nBest Hyperparameters:")
    for param, value in best_params.items():
        print(f"  {param}: {value}")

    return best_params

# ----------------------------
# Evaluate optimized model
# ----------------------------
def evaluate_model(best_params):
    global X_train, X_test, y_train, y_test

    print("\n" + "="*50)
    print("FINAL MODEL EVALUATION")
    print("="*50)

    best_rf = RandomForestClassifier(**best_params)
    best_rf.fit(X_train, y_train)

    y_pred = best_rf.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred)

    print(f"Test Accuracy: {test_accuracy:.6f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

    print("\nComparison with default Random Forest:")
    default_rf = RandomForestClassifier(random_state=42)
    default_rf.fit(X_train, y_train)
    default_pred = default_rf.predict(X_test)
    default_accuracy = accuracy_score(y_test, default_pred)

    print(f"Default RF Accuracy: {default_accuracy:.6f}")
    print(f"Optimized RF Accuracy: {test_accuracy:.6f}")
    print(f"Improvement: {test_accuracy - default_accuracy:.6f}")

if __name__ == "__main__":
    best_params = optimize_random_forest()
    evaluate_model(best_params)
    print("\n" + "="*50)
    print("OPTIMIZATION COMPLETE!")
    print("="*50) 


2025/09/11 09:53:43 AM, INFO, mealpy.swarm_based.PSO.OriginalPSO: OriginalPSO(epoch=5, pop_size=10, c1=2.05, c2=2.05, w=0.4)


Starting Random Forest optimization with Mealpy PSO...
Training set size: (2198, 1000)
Test set size: (550, 1000)
--------------------------------------------------


2025/09/11 09:54:19 AM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 1, Current best: -0.7024703230728324, Global best: -0.7024703230728324, Runtime: 16.56337 seconds
2025/09/11 09:54:36 AM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 2, Current best: -0.7024703230728324, Global best: -0.7024703230728324, Runtime: 17.03902 seconds
2025/09/11 09:55:02 AM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 3, Current best: -0.7024703230728324, Global best: -0.7024703230728324, Runtime: 26.40667 seconds
2025/09/11 09:55:24 AM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 4, Current best: -0.7024703230728324, Global best: -0.7024703230728324, Runtime: 22.32907 seconds
2025/09/11 09:55:41 AM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 5, Current best: -0.707473218079753, Global best: -0.707473218079753, Runtime: 16.25205 seconds


TypeError: 'Agent' object is not subscriptable